# E01 Iris：小样本分层验证与标签提交

> Issue: #9 / Epic #6 / 路线图 E01

## 你将学到
1. 小样本多分类为什么必须分层划分
2. 整数标签 vs 字符串标签如何正确生成 submission
3. 如何用 `wanglaoshi` 完成：环境检查 → EDA → 基线 → 提交校验

## 技术难点
- 每类只有 50 条，验证集波动大
- `target` 数字标签与 `species` 字符串标签提交格式不同
- 禁止对概率矩阵静默 argmax（见 B05 修复）


In [ ]:
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE
JE.no_warning()
V.show_env_info()


## 1. 加载 Iris，并同时保留数字标签与字符串标签


In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species'] = iris.target_names[iris.target]
print('shape =', df.shape)
print(df['target'].value_counts().sort_index())
print(df['species'].value_counts())
df.head()


## 2. EDA：用 DataAnalyzer 看分布与相关性


In [ ]:
from wanglaoshi import DataAnalyzer

analyzer = DataAnalyzer(df.drop(columns=['species']))  # 分析数值版
display(analyzer.missing_value_analysis())
display(analyzer.basic_statistics())
corr = analyzer.correlation_analysis()
display(corr)
# 如需热图（会返回 base64 字符串）
# analyzer.plot_correlation_heatmap()


## 3. 划分对比：随机 vs 分层

小样本下，非分层可能导致某折缺少某个类别。


In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold
from collections import Counter

X = df.drop(columns=['target', 'species'])
y = df['target']

print('=== KFold（不分层）各类出现情况 ===')
for i, (tr, te) in enumerate(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X)):
    print(i, Counter(y.iloc[te]))

print('=== StratifiedKFold ===')
for i, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X, y)):
    print(i, Counter(y.iloc[te]))


## 4. QuickBaseline 对比


In [ ]:
from sklearn.model_selection import train_test_split
from wanglaoshi import CompetitionTools as CT

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

baseline = CT.QuickBaseline()
results = baseline.run_all_models(X_train, y_train, X_valid, y_valid, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


## 5. 提交生成：数字标签与字符串标签两条路径

样本模板目标列名不一定叫 `target`。B01 修复后可自动推断。


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

best_name = comparison.iloc[0]['Model']
pred_int = results[best_name]['predictions']

out_dir = Path('outputs/01_iris')
out_dir.mkdir(parents=True, exist_ok=True)
gen = CT.SubmissionGenerator()

# ---- A) 数字标签模板 ----
sample_int = pd.DataFrame({'Id': np.arange(len(pred_int)), 'Prediction': y_valid.to_numpy()})
p1 = out_dir / 'sample_int.csv'
s1 = out_dir / 'sub_int.csv'
sample_int.to_csv(p1, index=False)
gen.create_submission(pred_int, str(p1), str(s1), prediction_type='label')
print('int validation:', gen.validate_submission(str(s1), str(p1)))

# ---- B) 字符串标签模板 ----
# 将整数预测映射回物种名
pred_str = iris.target_names[pred_int]
y_valid_str = iris.target_names[y_valid.to_numpy()]
sample_str = pd.DataFrame({'Id': np.arange(len(pred_str)), 'Species': y_valid_str})
p2 = out_dir / 'sample_str.csv'
s2 = out_dir / 'sub_str.csv'
sample_str.to_csv(p2, index=False)
gen.create_submission(pred_str, str(p2), str(s2), prediction_type='label')
print('str validation:', gen.validate_submission(str(s2), str(p2)))

# ---- C) 概率矩阵：必须显式 class_labels，禁止静默 argmax ----
best_model = results[best_name]['model']
if hasattr(best_model, 'predict_proba'):
    proba = best_model.predict_proba(X_valid)
    try:
        gen.create_submission(proba, str(p2), str(out_dir/'should_fail.csv'), prediction_type='proba')
        print('ERROR: 应该失败却成功了')
    except ValueError as e:
        print('按预期失败:', e)
    gen.create_submission(
        proba, str(p2), str(out_dir/'sub_from_proba.csv'),
        prediction_type='proba', class_labels=list(iris.target_names)
    )
    print('proba+class_labels validation:', gen.validate_submission(str(out_dir/'sub_from_proba.csv'), str(p2)))


## 6. 错分分析与局限

- Iris 近线性可分，基线分数通常很高，**不能**代表真实比赛难度
- 小验证集上 1–2 个错分就会让 Accuracy 波动明显
- 下一步可对比：只用 2 个特征、加入噪声、或换 StratifiedKFold 报告均值±方差


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_valid, pred_int, target_names=iris.target_names))
print(confusion_matrix(y_valid, pred_int))


## 检查清单
- [x] 分层划分
- [x] 数字/字符串两种提交
- [x] 概率提交需 class_labels
- [ ] 把 5 折 StratifiedKFold 的均值±方差记到实验笔记
